# Load deps

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

# Create OpenAI client

Check that the OpenAI client works

In [ ]:
openai_client = OpenAI()

# Plain LLMs lack our data

First, let's define a function to talk to the LLM:

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

This is our black box - text goes in, text comes out.

Let's test it:

In [ ]:
llm("Hey, what's up?")

Ask it a course-specific question:

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

This is different from a question like "how do I cook salmon?" - the LLM knows the answer because cooking salmon is common knowledge. But our courses are not in the training data.

# Adding context manually

More context can fix this. The FAQ website has questions and answers about our courses.

Copy some of that content into the prompt:

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

# Retrieval plus generation

RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. That search step is what gives the LLM the context it needs to answer correctly.

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

In code, it looks like this:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```
That's the entire architecture. It comes down to three components.

The pieces are search, the prompt, and the LLM:
-   search
-   prompt
-   LLM


The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, so search quality matters a lot for RAG.

The database and the LLM can be anything. Because each piece is independent, RAG stays flexible.
- for now, we use minsearch and then sqlitesearch for search, and OpenAI for the LLM
- To use Anthropic instead of OpenAI, you swap the LLM call.
- To use Elasticsearch instead of minsearch, you swap the search call. Nothing else changes.

## The Course FAQ Dataset

Before we build the RAG pipeline, let's get familiar with the data we'll use as our knowledge base.

The FAQ data is available as JSON from the DataTalks.Club website.

Let's fetch it:

In [ ]:
import requests
from pprint import pprint

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
pprint(courses_raw)

Let's fetch all the FAQ documents from all courses:

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

In [ ]:
print(len(documents))
pprint(documents[3])

## Using FAQ data

In the RAG pipeline, this dataset is our knowledge base:

1.  We index all the documents (the search step)
2.  When a student asks a question, we search the index
3.  The search returns the most relevant FAQ entries
4.  We give those entries to the LLM as context
5.  The LLM generates an answer based on the context

- The `question` and `answer` fields contain the text we'll search through.
- The `course` field lets us filter by course. For example, if a student asks about the data engineering course, we skip results from the ML course.
- The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

### A note on data preparation

- In our case, the data is already prepared in a convenient JSON format. So we don't need to do much to get it ready.
- Don't let that fool you. In reality, data preparation is often the most time-consuming part of building a RAG system. You may need to scrape websites, parse PDFs, and clean and chunk documents. That work isn't visible here.